In [1]:
import json
import requests
import os
import time
from random import uniform

In [2]:


# Setup
layer = 4220
step = 1  # Anzahl OBJECTIDs pro Anfrage
start_oid = 0
#max_oid = 5 #500  # ggf. vorher per Statistik abfragen
out_dir = f"layer_{layer}_geojson"
os.makedirs(out_dir, exist_ok=True)

base_url = f"https://datahub.uba.de/server/rest/services/VeLa/LK/MapServer/{layer}/query"


# OBJECTID-Spanne vorher automatisch bestimmen

r = requests.get(base_url, params={
    "where": "1=1",
    "f": "json",
    "outStatistics": json.dumps([{
        "statisticType": "max",
        "onStatisticField": "OBJECTID",
        "outStatisticFieldName": "max_id"
    }])
})
max_oid = int(r.json()["features"][0]["attributes"]["max_id"]) + 1

max_oid

276

In [3]:
def download_geojson_chunk(oid):
    where = f"OBJECTID = {oid}"
    params = {
        "where": where,
        "f": "geojson",
        "outFields": "*",
        "returnGeometry": "true"
    }
    try:
        r = requests.get(base_url, params=params, timeout=60)
        r.raise_for_status()
        out_path = os.path.join(out_dir, f"chunk_{oid}.geojson")
        with open(out_path, "wb") as f:
            f.write(r.content)
        print(f"✅ OBJECTID = {oid} gespeichert als {out_path}")
    except Exception as e:
        print(f"❌ Fehler bei OBJECTID = {oid}: {e}")

# Haupt-Loop
for oid in range(start_oid, max_oid):
    download_geojson_chunk(oid)
    time.sleep(uniform(0.5, 1.0))  # respectful pause


✅ OBJECTID = 0 gespeichert als layer_4220_geojson/chunk_0.geojson
✅ OBJECTID = 1 gespeichert als layer_4220_geojson/chunk_1.geojson
✅ OBJECTID = 2 gespeichert als layer_4220_geojson/chunk_2.geojson
✅ OBJECTID = 3 gespeichert als layer_4220_geojson/chunk_3.geojson
✅ OBJECTID = 4 gespeichert als layer_4220_geojson/chunk_4.geojson
✅ OBJECTID = 5 gespeichert als layer_4220_geojson/chunk_5.geojson
✅ OBJECTID = 6 gespeichert als layer_4220_geojson/chunk_6.geojson
✅ OBJECTID = 7 gespeichert als layer_4220_geojson/chunk_7.geojson
✅ OBJECTID = 8 gespeichert als layer_4220_geojson/chunk_8.geojson
✅ OBJECTID = 9 gespeichert als layer_4220_geojson/chunk_9.geojson
✅ OBJECTID = 10 gespeichert als layer_4220_geojson/chunk_10.geojson
✅ OBJECTID = 11 gespeichert als layer_4220_geojson/chunk_11.geojson
✅ OBJECTID = 12 gespeichert als layer_4220_geojson/chunk_12.geojson
✅ OBJECTID = 13 gespeichert als layer_4220_geojson/chunk_13.geojson
✅ OBJECTID = 14 gespeichert als layer_4220_geojson/chunk_14.geojson


In [4]:
!ogrinfo -al layer_4220_geojson/chunk_1.geojson

INFO: Open of `layer_4220_geojson/chunk_1.geojson'
      using driver `GeoJSON' successful.

Layer name: chunk_1
Geometry: Multi Polygon
Feature Count: 1
Extent: (12.932903, 52.345467) - (13.158774, 52.505993)
Layer SRS WKT:
GEOGCRS["WGS 84",
    DATUM["World Geodetic System 1984",
        ELLIPSOID["WGS 84",6378137,298.257223563,
            LENGTHUNIT["metre",1]]],
    PRIMEM["Greenwich",0,
        ANGLEUNIT["degree",0.0174532925199433]],
    CS[ellipsoidal,2],
        AXIS["geodetic latitude (Lat)",north,
            ORDER[1],
            ANGLEUNIT["degree",0.0174532925199433]],
        AXIS["geodetic longitude (Lon)",east,
            ORDER[2],
            ANGLEUNIT["degree",0.0174532925199433]],
    ID["EPSG",4326]]
Data axis to CRS axis mapping: 2,1
OBJECTID: Integer (0.0)
id: Integer (0.0)
Lärmpegelklasse: String (0.0)
source: String (0.0)
Shape_Length: Real (0.0)
Shape_Area: Real (0.0)
OGRFeature(chunk_1):1
  OBJECTID (Integer) = 1
  id (Integer) = 140100001
  Lärmpegelklasse (

In [5]:
import os
os.environ["OGR_GEOJSON_MAX_OBJ_SIZE"] = "800"  # in MB (z. B. 200 MB)

In [6]:
import geopandas as gpd
import os
import pandas as pd

folder = f"layer_{layer}_geojson"
files = sorted([f for f in os.listdir(folder) if f.endswith(".geojson")])

gdfs = []

for file in files:
    path = os.path.join(folder, file)
    try:
        gdf = gpd.read_file(path)
        gdfs.append(gdf)
    except Exception as e:
        print(f"❌ Error reading {file}: {e}")

# Merge all chunks
full_gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))

# Set CRS if not already present
if full_gdf.crs is None:
    full_gdf.set_crs(epsg=25833, inplace=True)  # or use correct CRS if different


#full_gdf.to_parquet("LK_BLR_road_Den_4210.parquet", index=False)

### try without cut
full_gdf.explode().to_file("LK_BLR_road_night_4220_exploded.fgb")


In [9]:
def fgb_to_pmtiles(polys_fgb, output_pmtiles, layer_name):
    import subprocess
    from pathlib import Path



    subprocess.run([
        "tippecanoe", "-o", output_pmtiles,
        f"--layer={layer_name}-polys",
        "--minimum-zoom=6", "--maximum-zoom=14",
        "--read-parallel",
        "--force",
        "--no-feature-limit", "--no-tile-size-limit", "--force-feature-limit",
        "--drop-densest-as-needed",
        "--drop-rate=0",
         "--coalesce", "--coalesce-densest-as-needed",
        str(Path(polys_fgb).resolve())
    ], check=True)


    print("✅ Combined PMTiles created")


In [ ]:
fgb_to_pmtiles(
    polys_fgb="LK_BLR_road_night_4220_exploded.fgb",
    output_pmtiles="laerm_4220_blr_night.pmtiles",
    layer_name="laerm_4220_blr_night"
    )

detected indexed FlatGeobuf: assigning feature IDs by sequence
624611 features, 403544968 bytes of geometry and attributes, 167506 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
  99.9%  14/8508/5486  


✅ Combined PMTiles created
